<a href="https://colab.research.google.com/github/robertbarcik/ADK-tutorial/blob/main/notebooks/13_a2a_protocol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 13 — A2A Protocol (the Side Step)

> **Where you are** — self-study side step, back on the OpenRouter key. The analogy that carries the whole module: **A2A is to agents what MCP is to tools** — a discovery-plus-call protocol, one level up.
> - **First met in M02/M10, used again here:** a second process on your machine (the remote agent) that your notebook starts and talks to over HTTP.

The course finale. Thirty minutes on something that isn't strictly ADK at all — the **A2A protocol** (Agent2Agent), the open standard for one agent calling another.

You've seen agents talk to tools (M02 — MCP is the protocol for that) and to each other inside one process (M06). This module is about agents talking to **other agents across a network** — a different process, a different team, possibly a different framework.

Where it stands (September 2026): Google launched A2A in 2025 and handed it to the Linux Foundation; the spec reached **v1.0 in March 2026**; since August 2026 A2A and MCP live under the same roof, the Agentic AI Foundation. Big names back it. Real-world deployments are still thin — so treat this as architecture worth knowing, not plumbing you'd bet a product on this year.

**What you'll build:**
- An ADK agent exposed over A2A via `to_a2a()` — a small web service.
- Its Agent Card, fetched from `/.well-known/agent-card.json`.
- A `RemoteA2aAgent` that consumes that service as if it were a local agent.

**What you'll leave with:**
- The four A2A nouns (Agent Card, Task, Message, Artifact).
- The A2A-vs-MCP mental model.
- An honest read on A2A's maturity.

**Running cost:** under \$0.01 (one OpenRouter call via the remote agent).

# Setup

In [1]:
!pip install -q google-adk==2.7.1 litellm==1.85.7 'a2a-sdk[http-server]==1.1.2' uvicorn python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null
print("✅ Packages installed.")

✅ Packages installed.


In [2]:
import os, sys, warnings
warnings.filterwarnings("ignore")
try: sys.stderr.fileno()
except Exception: sys.stderr = open(os.devnull, "w")

OPENROUTER_API_KEY = None
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
except Exception:
    try:
        from dotenv import load_dotenv; load_dotenv()
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
    except ImportError: pass
if not OPENROUTER_API_KEY:
    from getpass import getpass
    OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
print("✅ Environment ready.")

✅ Environment ready.


In [3]:
import subprocess, time, tempfile, shutil, signal, json, urllib.request, asyncio, uuid
import nest_asyncio; nest_asyncio.apply()

# ADK A2A pieces
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent, AGENT_CARD_WELL_KNOWN_PATH
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

print("✅ Imports successful.")

✅ Imports successful.


# The Four A2A Nouns

A2A reduces agent-to-agent communication to four concepts. Memorize these and the rest of the protocol is commentary.

| Noun | What it is | Analogue |
|---|---|---|
| **Agent Card** | JSON descriptor served at `/.well-known/agent-card.json` — identity, capabilities, skills, auth | OpenAPI spec for an agent |
| **Task** | Stateful, server-owned unit of work. Has an ID, status, history, artifacts | A GitHub Issue, roughly |
| **Message** | One turn in a task — user or agent, with typed parts (text/file/data) | A chat message |
| **Artifact** | Durable output of a task (reports, images, structured JSON) | An Issue's attachments |

**Skills** are the semantic menu a discovering agent sees — `translate_spanish`, `find_flights`, each with id/name/description/examples. **Capabilities** are protocol feature-flags — whether the agent supports streaming, push notifications, state history.

One teaching line worth remembering: *"skills are the restaurant's menu; capabilities are whether it does delivery."*

# A2A vs MCP

The dominant framing — and it's the right one:

> **MCP is agent↔tool. A2A is agent↔agent.**

What A2A adds that a tool call doesn't have:

- **Stateful, long-running tasks** — a task has a lifecycle (`working / input-required / completed`), not just one request and one response.
- **Peer symmetry** — every A2A agent is both client and server; no hierarchy is assumed.
- **Cross-framework interop** — an ADK agent calling an agent written in another framework, both speaking A2A.

Where the boundary blurs: an A2A agent with one synchronous skill is functionally indistinguishable from an MCP tool call. The distinction is the **interaction model**, not what's being called.

**The layered pattern to leave this module with:** an orchestrator uses **A2A** to reach specialist agents; each specialist internally uses **MCP** to call tools. The same shape you built in M02 + M06, with A2A as the network transport.

# Demo Part 1 — Expose an ADK Agent as A2A

The key line is one call:

```python
a2a_app = to_a2a(root_agent, port=8123)
```

The same wrapping move you've used all course — `FunctionTool` wrapped a function, `AgentTool` wrapped an agent for a *local* parent — and `to_a2a()` wraps an agent one step further: into a small web service that speaks A2A, so *any* client on the network can discover and call it. Under the hood it builds a Starlette app; you run it with `uvicorn` like any Python web app.

We'll write a small specialist agent to a temp directory, then launch it as a subprocess on `localhost:8123`. The agent does one thing: convert Celsius to Fahrenheit via a tool call.

In [4]:
# Write the specialist agent + server script to a temp dir
SRV_DIR = tempfile.mkdtemp(prefix="adk_m13_a2a_")
SRV_SCRIPT = os.path.join(SRV_DIR, "server.py")

SERVER_CODE = (
    "import os\n"
    "from dotenv import load_dotenv\n"
    "load_dotenv()\n\n"
    "from google.adk.agents import LlmAgent\n"
    "from google.adk.models.lite_llm import LiteLlm\n"
    "from google.adk.a2a.utils.agent_to_a2a import to_a2a\n"
    "import uvicorn\n\n"
    "def convert_c_to_f(celsius: float) -> dict:\n"
    "    'Convert Celsius to Fahrenheit.'\n"
    "    return {'celsius': celsius, 'fahrenheit': round(celsius * 9/5 + 32, 2)}\n\n"
    "root_agent = LlmAgent(\n"
    "    name='temperature_specialist',\n"
    "    model=LiteLlm(model='openrouter/openai/gpt-5.6-luna'),\n"
    "    description='Converts Celsius to Fahrenheit via convert_c_to_f tool.',\n"
    "    instruction='Convert temperatures using the tool. Return only the number.',\n"
    "    tools=[convert_c_to_f],\n"
    ")\n\n"
    "app = to_a2a(root_agent, host='localhost', port=8123)\n"
    "uvicorn.run(app, host='localhost', port=8123, log_level='error')\n"
)

with open(SRV_SCRIPT, "w") as f:
    f.write(SERVER_CODE)

# Launch server as subprocess
env = {**os.environ}
proc = subprocess.Popen(
    [sys.executable, SRV_SCRIPT],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    env=env,
)
time.sleep(7)  # give it time to start uvicorn + ADK
print(f"✅ A2A server running (PID {proc.pid}) on http://localhost:8123")

✅ A2A server running (PID 28896) on http://localhost:8123


## Fetch the Agent Card

The Agent Card is the A2A discovery mechanism. Served at `/.well-known/agent-card.json` (this path was renamed from `/.well-known/agent.json` in A2A v0.3.0 — a trap for anyone reading older blogs). A2A clients fetch it to learn:

- The agent's name, description, skills.
- The transport preference (JSON-RPC, gRPC, or REST).
- The protocol version, capabilities, supported modalities.
- Security schemes (OAuth, API key, mTLS, etc.).

In [5]:
with urllib.request.urlopen(f"http://localhost:8123{AGENT_CARD_WELL_KNOWN_PATH}") as r:
    card = json.loads(r.read().decode())

print(json.dumps(card, indent=2)[:2000])

{
  "name": "temperature_specialist",
  "description": "Converts Celsius to Fahrenheit via convert_c_to_f tool.",
  "supportedInterfaces": [
    {
      "url": "http://localhost:8123",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "version": "0.0.1",
  "capabilities": {
    "streaming": false,
    "pushNotifications": false
  },
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "skills": [
    {
      "id": "temperature_specialist",
      "name": "model",
      "description": "Converts Celsius to Fahrenheit via convert_c_to_f tool.",
      "tags": [
        "llm"
      ]
    },
    {
      "id": "temperature_specialist-convert_c_to_f",
      "name": "convert_c_to_f",
      "description": "Convert Celsius to Fahrenheit.",
      "tags": [
        "llm",
        "tools"
      ]
    }
  ]
}


The card is auto-generated from the agent's `name`, `description` and tools — each tool became a **skill**. Under `supportedInterfaces` the server tells clients where to call it (`url`), how (`protocolBinding: JSONRPC` — the default; gRPC and plain REST are the alternatives) and which spec it speaks (`protocolVersion: 1.0`).

In production you'd author the card more carefully — add `examples` to the skills, declare `capabilities.streaming: true` if your agent supports it, add security schemes and a signature. For a course demo, the auto-generated version is enough.

### 🎯 Mini-task

Author the card by hand and pass it via `to_a2a(agent_card=...)`: add `examples` to `skills`, declare `capabilities.streaming`. Does the demo still work with your hand-written card?

# Demo Part 2 — Consume the Agent as a `RemoteA2aAgent`

`RemoteA2aAgent` is the consuming side of the same idea: give it an Agent Card URL, it fetches the card, and from then on the remote agent behaves like a local `LlmAgent` object. Drop it into a runner's `agent=` argument, or into another agent's `sub_agents=` list — from the consuming side, it looks identical to a local agent.

One argument to always pass: `use_legacy=False`. The older executor (still the default) has known streaming bugs — duplicated user messages, remote output mis-labelled as thoughts. The new path fixes them.

In [6]:
remote = RemoteA2aAgent(
    name="remote_temp_agent",
    agent_card=f"http://localhost:8123{AGENT_CARD_WELL_KNOWN_PATH}",
    description="Remote temperature conversion specialist.",
    use_legacy=False,   # critical; skips the three known streaming bugs
)

print("✅ RemoteA2aAgent ready.")
print(f"   Pointing at: http://localhost:8123{AGENT_CARD_WELL_KNOWN_PATH}")

✅ RemoteA2aAgent ready.
   Pointing at: http://localhost:8123/.well-known/agent-card.json


In [7]:
# Call the remote agent locally via Runner — same API as any other LlmAgent
session_service = InMemorySessionService()
await session_service.create_session(app_name="m13", user_id="student", session_id="s1")
runner = Runner(agent=remote, app_name="m13", session_service=session_service)

msg = types.Content(role="user", parts=[types.Part(text="Convert 20 degrees Celsius to Fahrenheit.")])
print("USER: Convert 20 degrees Celsius to Fahrenheit.\n")

async for ev in runner.run_async(user_id="student", session_id="s1", new_message=msg):
    if ev.content and ev.content.parts:
        for p in ev.content.parts:
            if p.text and p.text.strip():
                print(f"[{ev.author}] {p.text.strip()[:200]}")
            if p.function_call:
                print(f"[tool_call] {p.function_call.name}")
            if p.function_response:
                print(f"[tool_resp] {p.function_response.response}")

USER: Convert 20 degrees Celsius to Fahrenheit.



[remote_temp_agent] 68


### 🔍 What just happened?

Read the event stream carefully. **The tool call (`convert_c_to_f`) executed on the remote server's side of the HTTP connection.** Your side sees it as a regular event — tool call, tool response, final text — just like if the agent were local.

This is what A2A buys you. From your agent's perspective, the specialist might as well be local. From the network's perspective, you just made an HTTP call to a separate process (or a separate machine, or a separate organization). The A2A protocol handles the translation.

**Remember the two multi-agent patterns?** `sub_agents` for transfer, `AgentTool` for consultant calls. `RemoteA2aAgent` plugs into either — it works as a child in `sub_agents=[RemoteA2aAgent(...)]`, or as the `agent=` argument of an `AgentTool`. Same composition patterns, A2A as the transport underneath.

### 🎯 Mini-tasks

1. **Cross-framework A2A.** Clone [`a2aproject/a2a-samples`](https://github.com/a2aproject/a2a-samples), run one of the non-ADK sample agents on port 8124, and consume it with a `RemoteA2aAgent` from this notebook. Does ADK talk to it cleanly?
2. **Two-agent orchestrator.** Keep the temperature specialist running, add a second remote specialist (string reversal), and build a local orchestrator with both in `sub_agents=`. Does it route correctly?

# Cleanup

In [8]:
proc.terminate()
try:
    proc.wait(timeout=3)
except subprocess.TimeoutExpired:
    proc.kill()
shutil.rmtree(SRV_DIR, ignore_errors=True)
print("✅ Server stopped, temp dir cleaned.")

✅ Server stopped, temp dir cleaned.


# Maturity, and Three Sharp Edges

The spec is real: **v1.0 since March 2026**, three interchangeable transports (JSON-RPC, gRPC, REST), signed Agent Cards, and since August 2026 the same foundation as MCP. The ecosystem is younger than the membership lists suggest — most member organizations are signatories, not shippers — and ADK's own A2A layer still prints `@a2a_experimental` warnings. Build against it for new work; don't move a production workload onto it this year.

Three things that bite:

1. **Pin ADK and `a2a-sdk` together.** ADK 2.7 accepts both the older 0.3 line and the current 1.x line, and the auto-generated card follows whichever is installed — two teams on different pins is the first thing to check when discovery fails. This notebook pins `a2a-sdk[http-server]==1.1.2`; the `[http-server]` extra is what `to_a2a()` needs to serve.
2. **`use_legacy=False`** on every `RemoteA2aAgent` — see above.
3. **An Agent Card is public, and a remote answer is input.** Whatever you put in `description` and `skills` is readable by anyone who can reach the server, and whatever a remote agent replies is data, not truth. Treat every cross-organization A2A response the way you treat a tool result from the open web: untrusted until checked.

# Key Takeaways — M13

- **Four A2A nouns**: Agent Card, Task, Message, Artifact.
- **A2A is agent↔agent; MCP is agent↔tool.** The layered pattern: orchestrator uses A2A; specialists use MCP.
- **`to_a2a(agent)`** exposes any ADK agent as an A2A service; **`RemoteA2aAgent(agent_card=...)`** consumes one from any process — and plugs into `sub_agents` or `AgentTool` like a local agent.
- **Agent Cards** live at `/.well-known/agent-card.json` — auto-generated by ADK, hand-authored for production.
- **`use_legacy=False`**, and pin `google-adk` and `a2a-sdk` together.
- **Maturity check**: architecture worth understanding, not production infrastructure yet; every remote answer is untrusted input.

# Course finale

Thirteen modules. From "what is an agent?" to "here's how agents talk to each other across organizations."

**Part 1 — the vendor-agnostic spine (M01–M10).** The four building blocks; four kinds of tools; state with scope prefixes; the one-line model swap; workflow agents (Sequential / Parallel / Loop); multi-agent via `sub_agents` and `AgentTool`; callbacks as your code around every step; memory that survives restarts; automated evaluation; deployment.

**Part 2 — Gemini unlocks (M11–M12).** Google Search grounding with real citations; long context with cached-token discounts; thinking budgets.

**Side step (M13).** A2A for agent-to-agent calls across frameworks.

**What to do next:** build something — a research orchestrator, a support agent that survives restarts. The course taught the mechanics; building teaches the rest. Keep an eye on the two protocols, MCP and A2A; they will outlast any framework. And watch the course repo: `DEMOS_BROKEN.md` tracks anything that stops working as APIs move.

Thanks for taking the course. Go build something.